In [0]:
# Configuration & Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import time

CATALOG = "fahrlytics"
BRONZE  = f"{CATALOG}.bronze"
SILVER  = f"{CATALOG}.silver"

spark.sql(f"USE CATALOG {CATALOG}")

VALID_CITIES = ["Berlin", "Munich", "Hamburg", "Frankfurt", "Cologne"]
VALID_VEHICLES = ["Standard", "Comfort", "Van/XL"]

print(f"Catalog : {CATALOG}")
print(f"Source  : {BRONZE}.drivers")
print(f"Target  : {SILVER}.drivers")

In [0]:
# Determine Load Strategy (Full vs. Incremental)
print("Evaluating load strategy...")

table_exists = spark.catalog.tableExists(f"{SILVER}.drivers")
is_empty = not table_exists or spark.table(f"{SILVER}.drivers").count() == 0

if is_empty:
    LOAD_TYPE = "full"
    df_bronze = spark.table(f"{BRONZE}.drivers")
    print("Initiating FULL load for drivers.")
else:
    LOAD_TYPE = "incremental"
    last_run = spark.sql(f"SELECT MAX(_processed_at) FROM {SILVER}.drivers").collect()[0][0]
    df_bronze = spark.table(f"{BRONZE}.drivers").filter(F.col("_ingested_at") > last_run)
    print(f"Initiating INCREMENTAL load for driver records ingested after: {last_run}")

input_count = df_bronze.count()
print(f"Incoming driver records: {input_count:,}")

if input_count == 0:
    print("No new driver records to process. Silver table is up to date.")
    dbutils.notebook.exit("No new driver records")

In [0]:
# Driver master is deduplicated by driver_id; latest ingested snapshot wins
window_spec = Window.partitionBy("driver_id").orderBy(
    F.col("_ingested_at").desc(),
    F.col("joining_date").desc()
)

df_deduped = (
    df_bronze
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

dedup_removed = input_count - df_deduped.count()
print(f"Deduplicated records: {df_deduped.count():,} (Removed {dedup_removed:,} duplicates)")

In [0]:
# Vehicle Standardization & Driver Status Validation
print("Applying German market standardization & business validation rules...")

df_validated = (
    df_deduped
    # 1. Standardize status (active vs. inactive)
    .withColumn("status_clean", F.lower(F.trim(F.col("status"))))
    .withColumn("is_active", F.col("status_clean") == "active")
    
    # 2. Standardize German vehicle categories
    .withColumn("vehicle_type_clean", 
        F.when(F.col("vehicle_type").isin("Standard", "Comfort", "Van/XL"), F.col("vehicle_type"))
        .when(F.lower(F.col("vehicle_type")).isin("sedan", "auto"), "Standard") # legacy fallback mapping
        .when(F.lower(F.col("vehicle_type")) == "suv", "Comfort")
        .otherwise("Standard")
    )
    
    # 3. Rating validation (bound between 1.0 and 5.0, impute nulls with default benchmark 4.5)
    .withColumn("rating_raw", F.col("rating"))
    .withColumn("is_valid_rating", (F.col("rating").isNotNull()) & (F.col("rating") >= 1.0) & (F.col("rating") <= 5.0))
    .withColumn("rating_clean", 
        F.when((F.col("rating") > 5.0) | (F.col("rating") < 1.0), 5.0)
        .when(F.col("rating").isNull(), 4.5)
        .otherwise(F.col("rating"))
    )
    
    # 4. Driver tenure & experience tiering
    .withColumn("tenure_days", F.datediff(F.current_date(), F.col("joining_date")))
    .withColumn("driver_experience_tier", 
        F.when(F.col("trips_completed") >= 3000, "Platinum / Pro")
        .when(F.col("trips_completed") >= 1000, "Gold")
        .when(F.col("trips_completed") >= 200, "Silver")
        .otherwise("Newbie")
    )
    .withColumn("_processed_at", F.current_timestamp())
)

In [0]:
# Create Silver Target Delta Table (if not exists)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER}.drivers (
        driver_id              STRING,
        name                   STRING,
        city                   STRING,
        vehicle_type_raw       STRING,
        vehicle_type           STRING,
        rating_raw             DOUBLE,
        rating                 DOUBLE,
        is_valid_rating        BOOLEAN,
        trips_completed        INT,
        joining_date           DATE,
        status_raw             STRING,
        is_active              BOOLEAN,
        tenure_days            INT,
        driver_experience_tier STRING,
        _ingested_at           TIMESTAMP,
        _processed_at          TIMESTAMP
    )
    USING DELTA
    PARTITIONED BY (city)
    COMMENT 'Silver verified driver profiles and tenure metrics'
""")

In [0]:
# Write or Upsert to Silver
print(f"Writing to {SILVER}.drivers (Mode: {LOAD_TYPE})...")
t0 = time.time()

df_to_write = (
    df_validated
    .select(
        "driver_id",
        "name",
        "city",
        F.col("vehicle_type").alias("vehicle_type_raw"),
        F.col("vehicle_type_clean").alias("vehicle_type"),
        "rating_raw",
        F.col("rating_clean").alias("rating"),
        "is_valid_rating",
        "trips_completed",
        "joining_date",
        F.col("status").alias("status_raw"),
        "is_active",
        "tenure_days",
        "driver_experience_tier",
        "_ingested_at",
        "_processed_at"
    )
)

if LOAD_TYPE == "full":
    (
        df_to_write.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("city")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER}.drivers")
    )
    print(f"Full driver load written: {df_to_write.count():,} rows")
else:
    target_drivers = DeltaTable.forName(spark, f"{SILVER}.drivers")
    (
        target_drivers.alias("target")
        .merge(
            df_to_write.alias("source"),
            "target.city = source.city AND target.driver_id = source.driver_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Incremental driver merge complete.")

print(f"Write duration: {round(time.time() - t0, 2)}s")

# File compaction
print("Running Delta OPTIMIZE...")
spark.sql(f"OPTIMIZE {SILVER}.drivers")
print("Delta OPTIMIZE completed.")

In [0]:
# Verification & Audit
print("=== FAHRLYTICS SILVER DRIVERS AUDIT ===\n")

df_verify = spark.table(f"{SILVER}.drivers")
total_drivers = df_verify.count()
print(f"Total drivers in silver : {total_drivers:,}")

print("\nVehicle Category Distribution:")
df_verify.groupBy("vehicle_type").count().orderBy("count", ascending=False).show()

print("Active vs. Inactive Drivers:")
df_verify.groupBy("is_active").count().show()

print("Driver Experience Tiers:")
df_verify.groupBy("driver_experience_tier").count().orderBy("count", ascending=False).show()

print("City Breakdown:")
df_verify.groupBy("city").count().orderBy("count", ascending=False).show()